In [1]:
import pandas as pd

In [2]:
from pathlib import Path
import os

# remonte jusqu'au dossier qui contient .git, puis s'y place
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").is_dir())
os.chdir(ROOT)
print("racine projet :", ROOT)

racine projet : /Users/benjaminscemama/dev/market-risk-control


In [3]:
%load_ext sql
%config SqlMagic.autopandas = True
%sql duckdb:///:memory:
%sql ATTACH IF NOT EXISTS 'data/risk.db' AS r (TYPE sqlite);

Connecting to 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

,Success


In [4]:
%%sql
-- Test configuration environnement
SELECT name FROM (SHOW ALL TABLES) WHERE database = 'r' ORDER BY name;

Running query in 'duckdb:///:memory:'

,name
0,mkt_forward_curve
1,mkt_spot_hourly
2,pos_snapshot
3,ref_contract
4,ref_customer
5,ref_site
6,trd_deal


In [157]:
df = %sql select * from r.trd_deal;
df_sorted = df.sort_values(["deal_id", "version"])
df_latest = df_sorted.drop_duplicates("deal_id", keep='last')
df_en_vigueur = df_latest[df_latest["status"] == "CONFIRMED"]

Running query in 'duckdb:///:memory:'

In [92]:
df['deal_id'].value_counts().value_counts()

count
1    8425
2     570
3       5
Name: count, dtype: int64

In [120]:
pred1 = pd.Series({
    "CONFIRMED": 0.90,
    "PENDING": 0.07,
    "CANCELLED": 0.03
})
pred1

CONFIRMED    0.90
PENDING      0.07
CANCELLED    0.03
dtype: float64

In [121]:
pred2 = pd.Series({
    "CONFIRMED": 0.93,
    "PENDING": 0.038,
    "CANCELLED": 0.032
})
pred2

CONFIRMED    0.930
PENDING      0.038
CANCELLED    0.032
dtype: float64

In [124]:
over_all_set = df["status"].value_counts(normalize = True)
over_all_set

status
CONFIRMED    0.926722
PENDING      0.042902
CANCELLED    0.030376
Name: proportion, dtype: float64

In [123]:
over_confirmed_set = df_latest["status"].value_counts(normalize = True)
over_confirmed_set

status
CONFIRMED    0.926333
PENDING      0.042778
CANCELLED    0.030889
Name: proportion, dtype: float64

In [111]:
def err(res, pred) : 
    return (res - pred).abs() / pred

In [122]:
print(f"{err(over_all_set, pred1)} \n")
print(err(over_confirmed_set, pred2))

status
CONFIRMED    0.029691
PENDING      0.387116
CANCELLED    0.012526
dtype: float64 

status
CONFIRMED    0.003943
PENDING      0.125731
CANCELLED    0.034722
dtype: float64


In [125]:
len(df_latest), df_latest["deal_id"].nunique()

(9000, 9000)

In [126]:
len(df_en_vigueur)

8337

In [134]:
df[df.duplicated(keep=False)]["deal_id"].nunique()

40

In [139]:
triples = df["deal_id"].value_counts()
triples = triples[triples == 3].index
df[df["deal_id"].isin(triples)].sort_values(["deal_id", "version"])

,deal_id,trade_date,trade_ts,commodity,direction,delivery_start,delivery_end,volume_mwh,price_eur_mwh,counterparty,book,status,version
313,D2600313,2025-06-05,2025-06-05 08:43:30,POWER,BUY,2026-06-01,2026-06-30,177.2,70.160,STATKRAFT,B2B_FR_POWER_HEDGE,CONFIRMED,1
9310,D2600313,2025-06-05,2025-06-06 08:43:30,POWER,BUY,2026-06-01,2026-06-30,185.7,68.693,STATKRAFT,B2B_FR_POWER_HEDGE,CONFIRMED,2
9556,D2600313,2025-06-05,2025-06-06 08:43:30,POWER,BUY,2026-06-01,2026-06-30,185.7,68.693,STATKRAFT,B2B_FR_POWER_HEDGE,CONFIRMED,2
1497,D2601497,2025-09-24,2025-09-24 11:53:37,POWER,SELL,2027-11-01,2027-11-30,265.3,86.853,UNIPER,B2B_FR_POWER_HEDGE,CONFIRMED,1
9545,D2601497,2025-09-24,2025-09-24 11:53:37,POWER,SELL,2027-11-01,2027-11-30,265.3,86.853,UNIPER,B2B_FR_POWER_HEDGE,CONFIRMED,1
9066,D2601497,2025-09-24,2025-09-25 11:53:37,POWER,SELL,2027-11-01,2027-11-30,240.6,87.908,UNIPER,B2B_FR_POWER_HEDGE,CONFIRMED,2
1680,D2601680,2025-09-22,2025-09-22 14:33:01,GAS,BUY,2027-02-01,2027-02-28,85.5,36.709,RWE,B2B_FR_STRUCT,CONFIRMED,1
9567,D2601680,2025-09-22,2025-09-22 14:33:01,GAS,BUY,2027-02-01,2027-02-28,85.5,36.709,RWE,B2B_FR_STRUCT,CONFIRMED,1
9001,D2601680,2025-09-22,2025-09-23 14:33:01,GAS,BUY,2027-02-01,2027-02-28,89.9,38.700,RWE,B2B_FR_STRUCT,CONFIRMED,2
1976,D2601976,2025-07-08,2025-07-08 17:34:22,GAS,BUY,2026-12-01,2026-12-31,216.2,38.439,ICE_ENDEX,B2B_FR_GAS_HEDGE,CONFIRMED,1


In [144]:
amend = df[df["deal_id"].isin(
    df["deal_id"].value_counts().loc[lambda s: s > 1].index
)].sort_values(["deal_id", "version"])

amend["ts"] = pd.to_datetime(amend["trade_ts"])
ecart = amend.groupby("deal_id")["ts"].agg(lambda s: (s.max() - s.min()).total_seconds() / 86400)
ecart.value_counts()

ts
1.0    540
0.0     35
Name: count, dtype: int64

In [188]:
df_tmp = df.copy(deep=True)
df_tmp["trade_date"] = pd.to_datetime(df_tmp["trade_date"])
df_tmp["date_2"] = pd.to_datetime(df_tmp["trade_ts"]).dt.normalize()

masque_different = df_tmp["trade_date"] != df_tmp["date_2"]

df_filtre = df[masque_different]

df_filtre

,deal_id,trade_date,trade_ts,commodity,direction,delivery_start,delivery_end,volume_mwh,price_eur_mwh,counterparty,book,status,version
9000,D2604989,2026-04-03,2026-04-04 16:24:55,GAS,SELL,2026-11-01,2027-01-31,211.8,35.225,EEX_CLEARED,B2B_FR_GAS_HEDGE,CONFIRMED,2
9001,D2601680,2025-09-22,2025-09-23 14:33:01,GAS,BUY,2027-02-01,2027-02-28,89.9,38.700,RWE,B2B_FR_STRUCT,CONFIRMED,2
9002,D2605511,2026-01-30,2026-01-31 10:40:45,POWER,BUY,2026-06-01,2026-06-30,333.7,49.700,VITOL,B2B_FR_POWER_HEDGE,CONFIRMED,2
9003,D2604854,2025-07-02,2025-07-03 14:53:34,GAS,SELL,2026-03-01,2026-03-31,661.9,31.564,VITOL,B2B_FR_GAS_HEDGE,CONFIRMED,2
9004,D2602053,2025-09-25,2025-09-26 09:56:15,POWER,SELL,2026-05-01,2026-05-31,557.0,73.451,ICE_ENDEX,B2B_FR_STRUCT,CONFIRMED,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9538,D2607408,2025-11-12,2025-11-13 10:01:58,POWER,BUY,2027-06-01,2028-05-31,398.0,71.325,TOTALENERGIES,B2B_FR_POWER_HEDGE,CONFIRMED,2
9539,D2602499,2025-12-01,2025-12-02 11:04:09,POWER,SELL,2026-03-01,2026-03-31,126.9,93.181,TOTALENERGIES,B2B_FR_POWER_HEDGE,CONFIRMED,2
9556,D2600313,2025-06-05,2025-06-06 08:43:30,POWER,BUY,2026-06-01,2026-06-30,185.7,68.693,STATKRAFT,B2B_FR_POWER_HEDGE,CONFIRMED,2
9564,D2601976,2025-07-08,2025-07-09 17:34:22,GAS,BUY,2026-12-01,2026-12-31,154.2,37.570,ICE_ENDEX,B2B_FR_GAS_HEDGE,CONFIRMED,2
